# Silver Layer — ERP: Customer Location (loc_a101)

Transforms the bronze `workspace.bronze.loc_a101` table into `workspace.silver.customer_info2` with meaningful, readable column names and cleaned data.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (`CID` -> `customer_id`, `CNTRY` -> `country`) |
| 2 | **Clean Customer ID** | Trimmed whitespace and removed dash from `customer_id` (e.g., `AW-00011000` -> `AW00011000`) |
| 3 | **Trim Whitespace** | Trimmed leading/trailing whitespace from all string columns |
| 3 | **Country Standardization** | Unified country abbreviations with full names: `USA`/`US` -> `United States`, `DE` -> `Germany` |
| 4 | **NULL Country Handling** | Replaced NULL and blank country values with `Unknown` |

---

## Column Mapping (Bronze to Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `CID` | `customer_id` | Renamed + Trimmed + Dash removed |
| `CNTRY` | `country` | Renamed + Trimmed + Standardized |

---

## Country Value Mapping

| Bronze Value(s) | Silver Value | Count |
|---|---|---|
| `United States` | `United States` | 3,391 |
| `USA` | `United States` | 2,591 |
| `US` | `United States` | 1,500 |
| `Germany` | `Germany` | 1,214 |
| `DE` | `Germany` | 566 |
| `Australia` | `Australia` | 3,591 |
| `United Kingdom` | `United Kingdom` | 1,913 |
| `France` | `France` | 1,810 |
| `Canada` | `Canada` | 1,571 |
| `NULL`, ` ` (spaces) | `Unknown` | 337 |

---

## Implementation

All transformations are implemented in **PySpark** (`pyspark.sql.functions`):
- `col()` — select and alias columns
- `trim()` — remove leading/trailing whitespace
- `when()` + `otherwise()` — conditional mapping for country standardization
- `df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable()` — write to silver layer

In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace

df = (spark.read.table("workspace.bronze.loc_a101")
    .select(
        regexp_replace(trim(col("CID")), "-", "").alias("customer_id"),
        when(trim(col("CNTRY")).isin("USA", "US"), "United States")
            .when(trim(col("CNTRY")) == "DE", "Germany")
            .when(trim(col("CNTRY")) == "", "Unknown")
            .when(col("CNTRY").isNull(), "Unknown")
            .otherwise(trim(col("CNTRY"))).alias("country"),
    )
)

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.customer_info2")

In [0]:
display(spark.read.table("workspace.silver.customer_info2").limit(10))